In [3]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached scipy-1.17.0-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached numpy-2.4.1-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ----------- ---------------------------- 2.4/8.1 MB 11.2 MB/s eta 0:00:01
   ----------------------- ---------------- 4.7/8.1 MB 11.9 MB/s eta 0:00:01
   ----------------------------------- ---- 7.1/8.1 MB 11.2 MB/s eta 0:00:01
   -------------------------------------- - 7.9/8.1 MB 10.4 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 9.1 MB/s  0:00:00
Using cached scipy-1.17.0-cp311-cp311-win_amd64.whl (36.4 MB)
Using cached numpy-2.4.1-cp311-cp311-win_amd64.whl (12.6 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   -------- ------------------------------- 1/5 [numpy]
   -------- -----------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Program Files\Blender Foundation\Blender 4.3\4.3\python\bin\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path

BASE_PATH = Path("data")
MERGED_CSV_PATH = BASE_PATH / "data" / "merged_aadhaar_data_sample.csv"


In [2]:
def load_csv_folder(folder_path):
    files = glob.glob(str(folder_path / "*.csv"))
    df_list = []
    for file in files:
        df_list.append(pd.read_csv(file))
    return pd.concat(df_list, ignore_index=True)


In [3]:
merged_df = pd.read_csv(MERGED_CSV_PATH)

# Keep downstream logic unchanged by reconstructing the same inputs
bio_df = merged_df[[
    'date', 'state', 'district', 'pincode',
    'bio_age_5_17', 'bio_age_17_'
]].copy()

demo_df = merged_df[[
    'date', 'state', 'district', 'pincode',
    'demo_age_5_17', 'demo_age_17_'
]].copy()

enrol_df = merged_df[[
    'date', 'state', 'district', 'pincode',
    'age_0_5', 'age_5_17', 'age_18_greater'
]].copy()

for df in [bio_df, demo_df, enrol_df]:
    df['date'] = pd.to_datetime(df['date'], format="%Y-%m-%d", errors="coerce")


In [4]:
bio_agg = bio_df.groupby(
    ['state', 'district', 'pincode'], as_index=False
)['bio_age_5_17'].sum()

enrol_agg = enrol_df.groupby(
    ['state', 'district', 'pincode'], as_index=False
)['age_0_5'].sum()


In [5]:
bcg_df = enrol_agg.merge(
    bio_agg, 
    on=['state', 'district', 'pincode'], 
    how='left'
).fillna(0)

bcg_df['BCG'] = bcg_df['age_0_5'] - bcg_df['bio_age_5_17']
bcg_df = bcg_df[bcg_df['BCG'] > 0]

bcg_df.sort_values('BCG', ascending=False).head(10)


,state,district,pincode,age_0_5,bio_age_5_17,BCG
30825,West Bengal,Dinajpur Uttar,733210,1734.0,0.0,1734.0
30824,West Bengal,Dinajpur Uttar,733207,1669.0,0.0,1669.0
6929,Gujarat,Banas Kantha,385535,1337.0,1.0,1336.0
8281,Haryana,Gurugram,122001,1166.0,0.0,1166.0
30823,West Bengal,Dinajpur Uttar,733202,1134.0,0.0,1134.0
6911,Gujarat,Banas Kantha,385001,1058.0,43.0,1015.0
5115,Bihar,Pashchim Champaran,845438,975.0,2.0,973.0
8399,Haryana,Nuh,122508,902.0,0.0,902.0
30821,West Bengal,Dinajpur Uttar,733134,823.0,0.0,823.0
30822,West Bengal,Dinajpur Uttar,733201,786.0,0.0,786.0


In [6]:
demo_agg = demo_df.groupby(
    ['state', 'district', 'pincode'], as_index=False
)['demo_age_5_17'].sum()


In [8]:
bur_df = bio_agg.merge(
    demo_agg,
    on=['state', 'district', 'pincode'],
    how='inner'
)

bur_df['BUR'] = bur_df['bio_age_5_17'] / (bur_df['demo_age_5_17'] + 1)
bur_df.sort_values('BUR').head(10)


,state,district,pincode,bio_age_5_17,demo_age_5_17,BUR
16082,Maharashtra,Gondiya *,441802,0.0,0.0,0.0
32957,West Bengal,West Midnapore,721257,0.0,0.0,0.0
32956,West Bengal,West Midnapore,721253,0.0,0.0,0.0
32955,West Bengal,West Midnapore,721252,0.0,0.0,0.0
32978,West Bengal,West Midnapore,721503,0.0,0.0,0.0
42,Andaman and Nicobar Islands,South Andaman,744210,0.0,0.0,0.0
32971,West Bengal,West Midnapore,721437,0.0,0.0,0.0
16145,Maharashtra,Jalgaon,425121,0.0,0.0,0.0
16160,Maharashtra,Jalgaon,425318,0.0,0.0,0.0
16176,Maharashtra,Jalna,431111,0.0,0.0,0.0


In [9]:
fafi_df = bcg_df.copy()

fafi_df['FAFI_rate'] = fafi_df['BCG'] / (fafi_df['age_0_5'] + 1)
fafi_df.sort_values('FAFI_rate', ascending=False).head(10)


,state,district,pincode,age_0_5,bio_age_5_17,BCG,FAFI_rate
30825,West Bengal,Dinajpur Uttar,733210,1734.0,0.0,1734.0,0.999424
30824,West Bengal,Dinajpur Uttar,733207,1669.0,0.0,1669.0,0.999401
8281,Haryana,Gurugram,122001,1166.0,0.0,1166.0,0.999143
30823,West Bengal,Dinajpur Uttar,733202,1134.0,0.0,1134.0,0.999119
8399,Haryana,Nuh,122508,902.0,0.0,902.0,0.998893
30821,West Bengal,Dinajpur Uttar,733134,823.0,0.0,823.0,0.998786
30822,West Bengal,Dinajpur Uttar,733201,786.0,0.0,786.0,0.998729
28767,Uttar Pradesh,Kushi Nagar,274304,779.0,0.0,779.0,0.998718
10940,Karnataka,Bengaluru Urban,560045,775.0,0.0,775.0,0.998711
10949,Karnataka,Bengaluru Urban,560068,742.0,0.0,742.0,0.998654


In [10]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min() + 1)

bis_df = fafi_df.merge(
    bur_df[['state','district','pincode','BUR']],
    on=['state','district','pincode'],
    how='left'
).fillna(0)

bis_df['BCG_norm'] = normalize(bis_df['BCG'])
bis_df['BUR_risk'] = 1 - bis_df['BUR']
bis_df['FAFI_norm'] = normalize(bis_df['FAFI_rate'])


In [11]:
bis_df['BIS'] = (
    0.4 * bis_df['BCG_norm'] +
    0.3 * bis_df['BUR_risk'] +
    0.3 * bis_df['FAFI_norm']
) * 100

bis_df.sort_values('BIS', ascending=False).head(10)


,state,district,pincode,age_0_5,bio_age_5_17,BCG,FAFI_rate,BUR,BCG_norm,BUR_risk,FAFI_norm,BIS
1848,West Bengal,Dinajpur Uttar,733210,1734.0,0.0,1734.0,0.999424,0.0,0.999423,1.0,0.497403,84.899036
1847,West Bengal,Dinajpur Uttar,733207,1669.0,0.0,1669.0,0.999401,0.0,0.961938,1.0,0.497392,83.399274
568,Haryana,Gurugram,122001,1166.0,0.0,1166.0,0.999143,0.0,0.671857,1.0,0.497262,71.792153
1846,West Bengal,Dinajpur Uttar,733202,1134.0,0.0,1134.0,0.999119,0.0,0.653403,1.0,0.497250,71.053612
574,Haryana,Nuh,122508,902.0,0.0,902.0,0.998893,0.0,0.519608,1.0,0.497137,65.698411
1844,West Bengal,Dinajpur Uttar,733134,823.0,0.0,823.0,0.998786,0.0,0.474048,1.0,0.497083,63.874434
1845,West Bengal,Dinajpur Uttar,733201,786.0,0.0,786.0,0.998729,0.0,0.452710,1.0,0.497055,63.020056
1678,Uttar Pradesh,Kushi Nagar,274304,779.0,0.0,779.0,0.998718,0.0,0.448674,1.0,0.497049,62.858407
733,Karnataka,Bengaluru Urban,560045,775.0,0.0,775.0,0.998711,0.0,0.446367,1.0,0.497045,62.766036
742,Karnataka,Bengaluru Urban,560068,742.0,0.0,742.0,0.998654,0.0,0.427336,1.0,0.497017,62.003927


In [12]:
def classify_alert(score):
    if score > 70:
        return "CRITICAL – Immediate Biometric Drive Required"
    elif score > 40:
        return "HIGH – Schedule School-Based Updates"
    else:
        return "MODERATE – Monitor & Awareness"

bis_df['Alert'] = bis_df['BIS'].apply(classify_alert)


In [13]:
final_output = bis_df[[
    'state', 'district', 'pincode',
    'BIS', 'Alert'
]].sort_values('BIS', ascending=False)

final_output.head(15)


,state,district,pincode,BIS,Alert
1848,West Bengal,Dinajpur Uttar,733210,84.899036,CRITICAL – Immediate Biometric Drive Required
1847,West Bengal,Dinajpur Uttar,733207,83.399274,CRITICAL – Immediate Biometric Drive Required
568,Haryana,Gurugram,122001,71.792153,CRITICAL – Immediate Biometric Drive Required
1846,West Bengal,Dinajpur Uttar,733202,71.053612,CRITICAL – Immediate Biometric Drive Required
574,Haryana,Nuh,122508,65.698411,HIGH – Schedule School-Based Updates
1844,West Bengal,Dinajpur Uttar,733134,63.874434,HIGH – Schedule School-Based Updates
1845,West Bengal,Dinajpur Uttar,733201,63.020056,HIGH – Schedule School-Based Updates
1678,Uttar Pradesh,Kushi Nagar,274304,62.858407,HIGH – Schedule School-Based Updates
733,Karnataka,Bengaluru Urban,560045,62.766036,HIGH – Schedule School-Based Updates
742,Karnataka,Bengaluru Urban,560068,62.003927,HIGH – Schedule School-Based Updates


In [14]:
from sklearn.ensemble import IsolationForest

features = bis_df[[
    'BCG', 'FAFI_rate', 'BUR'
]].fillna(0)

model = IsolationForest(
    n_estimators=100,
    max_samples='auto',
    contamination=0.02,
    random_state=42
)

bis_df['AnomalyScore'] = model.fit_predict(features)
bis_df['AnomalyFlag'] = bis_df['AnomalyScore'] == -1

bis_df[bis_df['AnomalyFlag']].sort_values('BIS', ascending=False).head(10)


,state,district,pincode,age_0_5,bio_age_5_17,BCG,FAFI_rate,BUR,BCG_norm,BUR_risk,FAFI_norm,BIS,Alert,AnomalyScore,AnomalyFlag
1848,West Bengal,Dinajpur Uttar,733210,1734.0,0.0,1734.0,0.999424,0.0,0.999423,1.0,0.497403,84.899036,CRITICAL – Immediate Biometric Drive Required,-1,True
1847,West Bengal,Dinajpur Uttar,733207,1669.0,0.0,1669.0,0.999401,0.0,0.961938,1.0,0.497392,83.399274,CRITICAL – Immediate Biometric Drive Required,-1,True
568,Haryana,Gurugram,122001,1166.0,0.0,1166.0,0.999143,0.0,0.671857,1.0,0.497262,71.792153,CRITICAL – Immediate Biometric Drive Required,-1,True
1846,West Bengal,Dinajpur Uttar,733202,1134.0,0.0,1134.0,0.999119,0.0,0.653403,1.0,0.497250,71.053612,CRITICAL – Immediate Biometric Drive Required,-1,True
574,Haryana,Nuh,122508,902.0,0.0,902.0,0.998893,0.0,0.519608,1.0,0.497137,65.698411,HIGH – Schedule School-Based Updates,-1,True
1844,West Bengal,Dinajpur Uttar,733134,823.0,0.0,823.0,0.998786,0.0,0.474048,1.0,0.497083,63.874434,HIGH – Schedule School-Based Updates,-1,True
1845,West Bengal,Dinajpur Uttar,733201,786.0,0.0,786.0,0.998729,0.0,0.452710,1.0,0.497055,63.020056,HIGH – Schedule School-Based Updates,-1,True
1678,Uttar Pradesh,Kushi Nagar,274304,779.0,0.0,779.0,0.998718,0.0,0.448674,1.0,0.497049,62.858407,HIGH – Schedule School-Based Updates,-1,True
733,Karnataka,Bengaluru Urban,560045,775.0,0.0,775.0,0.998711,0.0,0.446367,1.0,0.497045,62.766036,HIGH – Schedule School-Based Updates,-1,True
461,Gujarat,Banas Kantha,385535,1337.0,1.0,1336.0,0.998505,0.5,0.769896,0.5,0.496942,60.704104,HIGH – Schedule School-Based Updates,-1,True


In [15]:
import pickle

# Save the trained model
model_path = 'anomaly_detection_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"Model saved to {model_path}")


Model saved to anomaly_detection_model.pkl
